In [46]:
mkdir -p ./data/{01_RAW,02_CLEANED,03_TRAINING,04_VALIDATION,05_DEMO,00_MODEL}

In [50]:
import pandas as pd

# INPUT RAW
raw = pd.read_csv("./data/01_RAW/filmwebplus", encoding="utf-8-sig")
raw = raw[["review", "sentiment"]]
raw = raw.drop_duplicates()

# -> ./data/02_CLEANED.csv
raw.to_csv("./data/02_CLEANED/data.csv", index=False, encoding="utf-8-sig")

# -> ./data/03_TRAINING.csv
cleaned = pd.read_csv("./data/02_CLEANED/data.csv", encoding="utf-8-sig")
# Take only random 75% of sentiment 0 (negative) AND 2 (positive)
cleaned = cleaned[cleaned["sentiment"].isin([0, 2])]
negative = cleaned[cleaned["sentiment"] == 0]
positive = cleaned[cleaned["sentiment"] == 2]

n = min(len(negative), len(positive))

cleaned = pd.concat([
    negative.sample(n=n, random_state=42),
    positive.sample(n=n, random_state=42)
])

training = cleaned.sample(frac=0.75, random_state=42)
training.to_csv("./data/03_TRAINING/data.csv", index=False, encoding="utf-8-sig")

# -> ./data/04_VALIDATION.csv
validation = cleaned.drop(training.index)
validation.to_csv("./data/04_VALIDATION/data.csv", index=False, encoding="utf-8-sig")

# -> ./data/05_DEMO.csv
demo = raw[raw["sentiment"] == -1]
demo.to_csv("./data/05_DEMO/data.csv", index=False, encoding="utf-8-sig")

In [63]:
import pandas as pd
from sklearn.tree import DecisionTreeClassifier
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix
from sklearn.linear_model import Perceptron

import joblib

#
dataset = pd.read_csv("./data/03_TRAINING/data.csv", encoding="UTF-8")

# Vectorize text
vectorizer = TfidfVectorizer(
    max_features=2000,
    lowercase=True,
    ngram_range=(1, 2),
    min_df=2
)

# Create FEATURE
X = vectorizer.fit_transform(dataset["review"])

# Create LABEL
y = dataset["sentiment"]

# Initalize
model = Perceptron(random_state=42)
#model = LogisticRegression(random_state=42)
#model = DecisionTreeClassifier(random_state=42)

model.fit(X, y)
joblib.dump(model, "./data/00_MODEL/reviewsentiment.joblib")
joblib.dump(vectorizer, "./data/00_MODEL/vectorizer.joblib")

print("Successfully trained!")




validation = pd.read_csv("./data/04_VALIDATION/data.csv", encoding="utf-8-sig")
X_validation = vectorizer.transform(validation["review"])
y_validation = validation["sentiment"]

# Predict sentiment
predictions = model.predict(X_validation)

# -> VALIDATION RESULTS
accuracy = accuracy_score(y_validation, predictions)

correct = (y_validation == predictions).sum()
incorrect = (y_validation != predictions).sum()
total = len(y_validation)

print("\n========== VALIDATION RESULTS ==========")

print(f"Total reviews: {total}")
print(f"Correct predictions: {correct}")
print(f"Incorrect predictions: {incorrect}")
print(f"Accuracy: {accuracy * 100:.2f}%")

print("\n========== SUMMARY ==========")

print(f"The model correctly classified {correct} out of {total} reviews.")
print(f"Overall accuracy: {accuracy * 100:.2f}%")


Successfully trained!

========== VALIDATION RESULTS ==========
Total reviews: 1542
Correct predictions: 1301
Incorrect predictions: 241
Accuracy: 84.37%

========== SUMMARY ==========
The model correctly classified 1301 out of 1542 reviews.
Overall accuracy: 84.37%


Test na opiniach z datasetu przewidzianego do DEMO

In [64]:
import pandas as pd
import joblib

# Load model and vectorizer
model = joblib.load("./data/00_MODEL/reviewsentiment.joblib")
vectorizer = joblib.load("./data/00_MODEL/vectorizer.joblib")

# Load demo dataset
demo = pd.read_csv("./data/05_DEMO/data.csv", encoding="utf-8-sig")

# Vectorize reviews
X_demo = vectorizer.transform(demo["review"])

# Predict sentiment
predictions = model.predict(X_demo)

# Add predictions to dataframe
demo["predicted_sentiment"] = predictions

# Random 5 reviews
demo_sample = demo.sample(n=5, random_state=42)

for index, row in demo_sample.iterrows():
    actual = "NEGATIVE" if row["sentiment"] == 0 else "POSITIVE" if row["sentiment"] == 2 else "UNKNOWN"
    predicted = "NEGATIVE" if row["predicted_sentiment"] == 0 else "POSITIVE"

    print(f"REVIEW: {row['review']}")
    print(f"ACTUAL: {actual}")
    print(f"PREDICTED: {predicted}")
    print("-" * 80)

REVIEW: "Zejście" jest jednym z lepszych horrorów ostatnich lat. Przedstawia on historię sześciu miłośniczek sportów ekstremalnych i koleżanek - Sarah, Sam, Beth, Holly, Juno i Rebbeca - których przyjaźń została wystawiona na próbę po wypadku samochodowym, w którym zginęła córka i mąż Sary. W tym wypadku nie tylko Sarah coś straciła...
Mija rok. Juno namawia piątkę dziewczyn do wzięcia udziału w wyprawie, mówiąc, że to dobrze zrobi Sarze. Wszystkie się zgodziły i były zachwycone pięknem jaskiń, w których się znalazły. Podczas przeciskania się miedzy skałami Sarah o mało nie ginie, a wyjście na powierzchnię zostaje zamknięte. Żadna z sześciu przyjaciółek nie zdaje sobie sprawy, że będąc cztery kilometry pod ziemią, ich największym zmartwieniem nie będzie brak tlenu. Kilka metrów dalej czyhają na nie ohydne i krwiożercze stwory, które przecież muszą coś jeść...
Zdjęcia są chyba najsilniejszą stroną "Zejścia". Piękne widoki potrafią powalić na kolana. Aż trudno uwierzyć, że cała jaskinia 

Test na krótkich opiniach (innych niż z datasetu)

In [67]:
opinie = [
    "Świetny film, bardzo dobra fabuła i rewelacyjna gra aktorska",
    "Nie mogłem oderwać wzroku od ekranu, zdecydowanie jeden z najlepszych filmów jakie widziałem",
    "Film był naprawdę zabawny i ciekawy, chętnie obejrzałbym go ponownie",
    "Strasznie nudny film, fabuła była przewidywalna i kompletnie mnie nie zainteresowała",
    "Nie polecam, film był zdecydowanie za długi i nic ciekawego się w nim nie wydarzyło",
    "Fatalna produkcja, kiepskie efekty specjalne i brak jakiejkolwiek sensownej historii"
]

# 0 - negative
# 2 - positive

vectorized_opinie = vectorizer.transform(opinie)
predictions = model.predict(vectorized_opinie)

for opinia, prediction in zip(opinie, predictions):
    etykieta = "positive" if prediction == 2 else "negative"
    print(f"'{opinia}' -> {prediction} ({etykieta})")

'Świetny film, bardzo dobra fabuła i rewelacyjna gra aktorska' -> 0 (negative)
'Nie mogłem oderwać wzroku od ekranu, zdecydowanie jeden z najlepszych filmów jakie widziałem' -> 2 (positive)
'Film był naprawdę zabawny i ciekawy, chętnie obejrzałbym go ponownie' -> 2 (positive)
'Strasznie nudny film, fabuła była przewidywalna i kompletnie mnie nie zainteresowała' -> 0 (negative)
'Nie polecam, film był zdecydowanie za długi i nic ciekawego się w nim nie wydarzyło' -> 0 (negative)
'Fatalna produkcja, kiepskie efekty specjalne i brak jakiejkolwiek sensownej historii' -> 0 (negative)
